In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, FloatType

from delta import *

import logging

logging.getLogger("py4j").setLevel(logging.DEBUG)

In [2]:
# Create SparkSession
spark = (
    SparkSession
    .builder
    .master("local[*]")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

your 131072x1 screen size is bogus. expect trouble
26/04/28 20:10:32 WARN Utils: Your hostname, DESKTOP-J27O5E0 resolves to a loopback address: 127.0.1.1; using 172.22.146.234 instead (on interface eth0)
26/04/28 20:10:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/william/.cache/pypoetry/virtualenvs/spark-delta-EWHD1nIR-py3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/william/.ivy2/cache
The jars for the packages stored in: /home/william/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e2eb3cec-4816-476b-a0bc-0ef9040afdb4;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 187ms :: artifacts dl 7ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   

In [3]:
import os
import shutil

path = "spark-warehouse/policies"
if os.path.exists(path):
    shutil.rmtree(path)

spark.sql("DROP TABLE IF EXISTS policies")

DataFrame[]

In [4]:
spark.sql("""
CREATE TABLE policies (
  policy_id INT,
  customer_name STRING,
  policy_type STRING,
  premium_value DECIMAL(10,2),
  status STRING,
  created_at TIMESTAMP
)
USING DELTA
""")

DataFrame[]

In [5]:
spark.sql("""
INSERT INTO policies VALUES
(1, 'Ana Silva', 'AUTO', 120.50, 'ACTIVE', current_timestamp()),
(2, 'Carlos Lima', 'HOME', 300.00, 'ACTIVE', current_timestamp())
""")

spark.sql("""
SELECT * FROM policies
""").show(truncate=False)

26/04/28 20:10:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-------------+-----------+-------------+------+--------------------------+
|policy_id|customer_name|policy_type|premium_value|status|created_at                |
+---------+-------------+-----------+-------------+------+--------------------------+
|2        |Carlos Lima  |HOME       |300.00       |ACTIVE|2026-04-28 20:10:50.418477|
|1        |Ana Silva    |AUTO       |120.50       |ACTIVE|2026-04-28 20:10:50.418477|
+---------+-------------+-----------+-------------+------+--------------------------+



In [6]:
spark.sql("""
UPDATE policies
SET status = 'CANCELED'
WHERE policy_id = 1
""")

spark.sql("""
SELECT * FROM policies
""").show(truncate=False)

+---------+-------------+-----------+-------------+--------+--------------------------+
|policy_id|customer_name|policy_type|premium_value|status  |created_at                |
+---------+-------------+-----------+-------------+--------+--------------------------+
|2        |Carlos Lima  |HOME       |300.00       |ACTIVE  |2026-04-28 20:10:50.418477|
|1        |Ana Silva    |AUTO       |120.50       |CANCELED|2026-04-28 20:10:50.418477|
+---------+-------------+-----------+-------------+--------+--------------------------+



In [7]:
spark.sql("""
DELETE FROM policies
WHERE policy_id = 2
""")

spark.sql("""
SELECT * FROM policies
""").show(truncate=False)

+---------+-------------+-----------+-------------+--------+--------------------------+
|policy_id|customer_name|policy_type|premium_value|status  |created_at                |
+---------+-------------+-----------+-------------+--------+--------------------------+
|1        |Ana Silva    |AUTO       |120.50       |CANCELED|2026-04-28 20:10:50.418477|
+---------+-------------+-----------+-------------+--------+--------------------------+



In [8]:
spark.sql("""
DESCRIBE HISTORY policies
""").show(truncate=False)

+-------+-----------------------+------+--------+------------+----------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation   |operationParameters                                                                           |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                                                                                                           